## Spark for Teams 

### Setup

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import FloatType, IntegerType
import json
import os

In [ ]:
spark = SparkSession.builder \
    .appName("LaLiga_EDA_Bronze_Teams") \
    .master("local[*]") \
    .getOrCreate()

In [ ]:
spark

In [6]:
spark.sparkContext.setLogLevel("WARN")

BRONZE_TEAMS = "../data/1_bronze/teams"

In [7]:
print(f"✅ SparkSession aktif: {spark.sparkContext.appName}")
print(f"📁 Bronze path: {BRONZE_TEAMS}")

✅ SparkSession aktif: LaLiga_EDA_Bronze_Teams
📁 Bronze path: ../data/1_bronze/teams


In [8]:
# List semua file
for f in sorted(os.listdir(BRONZE_TEAMS)):
    if f.endswith(".json"):
        print(f"   📄 {f}")

   📄 teams_attacking.json
   📄 teams_defending.json
   📄 teams_misc.json
   📄 teams_passing.json
   📄 teams_pressing.json
   📄 teams_sequences.json


### Load JSON file to Spark DataFrame

In [9]:
df_attacking  = spark.read.json(f"{BRONZE_TEAMS}/teams_attacking.json",  multiLine=True)
df_defending  = spark.read.json(f"{BRONZE_TEAMS}/teams_defending.json",  multiLine=True)
df_passing    = spark.read.json(f"{BRONZE_TEAMS}/teams_passing.json",    multiLine=True)
df_pressing   = spark.read.json(f"{BRONZE_TEAMS}/teams_pressing.json",   multiLine=True)
df_sequences  = spark.read.json(f"{BRONZE_TEAMS}/teams_sequences.json",  multiLine=True)
df_misc       = spark.read.json(f"{BRONZE_TEAMS}/teams_misc.json",       multiLine=True)

In [10]:
datasets = {
    "attacking" : df_attacking,
    "defending" : df_defending,
    "passing"   : df_passing,
    "pressing"  : df_pressing,
    "sequences" : df_sequences,
    "misc"      : df_misc,
}
print("✅ Semua file berhasil di-load\n")
for name, df in datasets.items():
    print(f"  {name:12s} → {df.count()} baris, {len(df.columns)} kolom")

✅ Semua file berhasil di-load

  attacking    → 20 baris, 9 kolom
  defending    → 20 baris, 10 kolom
  passing      → 20 baris, 17 kolom
  pressing     → 20 baris, 9 kolom
  sequences    → 20 baris, 10 kolom
  misc         → 20 baris, 13 kolom


In [ ]:
# ────────────────────────────────────────────────
# CELL 3 — Quick Inspection per Kategori
# ────────────────────────────────────────────────
for name, df in datasets.items():
    print(f"\n{'='*55}")
    print(f"  📊 {name.upper()}")
    print(f"{'='*55}")
    print(f"Shape: ({df.count()}, {len(df.columns)})")
    print(f"\n🔹 Schema:")
    df.printSchema()
    print(f"🔹 Preview (3 baris):")
    df.show(3, truncate=False)

In [13]:
# ────────────────────────────────────────────────
# CELL 4 — Cek Jumlah Klub per Kategori
# ────────────────────────────────────────────────
print("📋 Jumlah klub per kategori:\n")
for name, df in datasets.items():
    n = df.select("club").distinct().count()
    status = "✅" if n == 20 else "⚠️ KURANG"
    print(f"  {name:12s} → {n} klub {status}")
# Referensi 20 klub
with open("../data/1_bronze/teams_list.txt") as f:
    teams_list = json.load(f)
print(f"\n📌 20 Klub Referensi:\n{teams_list}")

📋 Jumlah klub per kategori:

  attacking    → 20 klub ✅
  defending    → 20 klub ✅
  passing      → 20 klub ✅
  pressing     → 20 klub ✅
  sequences    → 20 klub ✅
  misc         → 20 klub ✅

📌 20 Klub Referensi:
['Alaves', 'Athletic Bilbao', 'Atlético Madrid', 'Barcelona', 'Celta Vigo', 'Elche', 'Espanyol', 'Getafe', 'Girona', 'Levante', 'Mallorca', 'Osasuna', 'Rayo Vallecano', 'Real Betis', 'Real Madrid', 'Real Oviedo', 'Real Sociedad', 'Sevilla', 'Valencia', 'Villarreal']


In [14]:
# ────────────────────────────────────────────────
# CELL 5 — Cek Missing Values
# ────────────────────────────────────────────────
print("🔍 Missing Values per Kategori:\n")
for name, df in datasets.items():
    total_missing = 0
    missing_cols = []
    for col_name in df.columns:
        null_count = df.filter(F.col(col_name).isNull()).count()
        if null_count > 0:
            total_missing += null_count
            missing_cols.append(f"{col_name}({null_count})")
    
    status = "✅ Tidak ada" if total_missing == 0 else f"⚠️ {total_missing} null"
    print(f"  {name:12s} → {status}")
    if missing_cols:
        print(f"               {missing_cols}")

🔍 Missing Values per Kategori:

  attacking    → ✅ Tidak ada
  defending    → ✅ Tidak ada
  passing      → ✅ Tidak ada
  pressing     → ✅ Tidak ada
  sequences    → ✅ Tidak ada
  misc         → ✅ Tidak ada


In [15]:
# ────────────────────────────────────────────────
# CELL 6 — Cek Konsistensi Nama Klub
# ────────────────────────────────────────────────
ref_set = set(teams_list)
print("🔍 Pengecekan nama klub vs referensi:\n")
for name, df in datasets.items():
    clubs = set(df.select("club").rdd.flatMap(lambda x: x).collect())
    extra = clubs - ref_set
    missing = ref_set - clubs
    if extra or missing:
        print(f"  ⚠️  {name}:")
        if extra:   print(f"       Extra   : {extra}")
        if missing: print(f"       Missing : {missing}")
    else:
        print(f"  ✅  {name}: semua 20 klub cocok")

🔍 Pengecekan nama klub vs referensi:

  ✅  attacking: semua 20 klub cocok
  ✅  defending: semua 20 klub cocok
  ✅  passing: semua 20 klub cocok
  ✅  pressing: semua 20 klub cocok
  ✅  sequences: semua 20 klub cocok
  ✅  misc: semua 20 klub cocok


In [16]:
# ────────────────────────────────────────────────
# CELL 7 — Identifikasi Kolom Persentase (String)
# ────────────────────────────────────────────────
print("📌 Kolom berformat string '%' yang perlu di-convert:\n")
for name, df in datasets.items():
    pct_cols = [
        field.name for field in df.schema.fields
        if field.dataType.simpleString() == "string" and field.name != "club"
    ]
    if pct_cols:
        print(f"  {name:12s} → {pct_cols}")
    else:
        print(f"  {name:12s} → tidak ada")

📌 Kolom berformat string '%' yang perlu di-convert:

  attacking    → ['conversion_pct']
  defending    → ['aerial_duels_won_pct', 'avg_possession_pct', 'ground_duels_won_pct']
  passing      → ['avg_possession_pct', 'crosses_pct', 'direction_bwd_pct', 'direction_fwd_pct', 'direction_left_pct', 'direction_right_pct', 'final_third_pct', 'passes_pct']
  pressing     → ['high_turnovers_shot_pct']
  sequences    → tidak ada
  misc         → tidak ada


In [17]:
# ────────────────────────────────────────────────
# CELL 8 — Fungsi Parse Persentase (String → Float)
# ────────────────────────────────────────────────
def parse_pct_columns(df):
    """
    Convert semua kolom string yang berisi '%' → float.
    '13.73%' → 0.1373
    """
    for field in df.schema.fields:
        if field.dataType.simpleString() == "string" and field.name != "club":
            df = df.withColumn(
                field.name,
                (
                    F.regexp_replace(F.col(field.name), "%", "")
                    .cast(FloatType()) / 100
                )
            )
    return df

In [18]:
# Terapkan ke semua dataset
datasets_parsed = {}
for name, df in datasets.items():
    datasets_parsed[name] = parse_pct_columns(df)
    print(f"✅ {name} parsed")
# Preview
print("\n🔎 Preview attacking setelah parsing:")
datasets_parsed["attacking"].show(3, truncate=False)
datasets_parsed["attacking"].printSchema()

✅ attacking parsed
✅ defending parsed
✅ passing parsed
✅ pressing parsed
✅ sequences parsed
✅ misc parsed

🔎 Preview attacking setelah parsing:
+-----------+-------------------+-----+-----------+------+-----+---+-----+-----------+
|club       |conversion_pct     |goals|goals_vs_xg|played|shots|sot|xg   |xg_per_shot|
+-----------+-------------------+-----+-----------+------+-----+---+-----+-----------+
|Barcelona  |0.1372999954223633 |95   |8.38       |38    |692  |257|86.62|0.13       |
|Real Madrid|0.11579999923706055|77   |-1.0       |38    |665  |253|78.0 |0.12       |
|Villarreal |0.1581999969482422 |72   |12.96      |38    |455  |172|59.04|0.13       |
+-----------+-------------------+-----+-----------+------+-----+---+-----+-----------+
only showing top 3 rows
root
 |-- club: string (nullable = true)
 |-- conversion_pct: double (nullable = true)
 |-- goals: long (nullable = true)
 |-- goals_vs_xg: double (nullable = true)
 |-- played: long (nullable = true)
 |-- shots: long (null

In [19]:
# ────────────────────────────────────────────────
# CELL 9 — Merge / Join Semua Kategori
# ────────────────────────────────────────────────
# Strategi: inner join on 'club', drop kolom duplikat
df_merged = datasets_parsed["attacking"]
merge_order = ["defending", "passing", "pressing", "sequences", "misc"]
for name in merge_order:
    df_right = datasets_parsed[name]
    
    # Kolom yang sudah ada di df_merged (selain 'club')
    overlap = [c for c in df_right.columns if c in df_merged.columns and c != "club"]
    
    # Drop kolom duplikat dari df_right
    df_right_clean = df_right.drop(*overlap)
    
    df_merged = df_merged.join(df_right_clean, on="club", how="inner")
    print(f"  Merge {name:12s} → kolom: {len(df_merged.columns)}, baris: {df_merged.count()}")
print(f"\n✅ Final merged:")
print(f"   Jumlah klub  : {df_merged.count()}")
print(f"   Jumlah kolom : {len(df_merged.columns)}")
print(f"   Kolom        : {df_merged.columns}")

  Merge defending    → kolom: 17, baris: 20
  Merge passing      → kolom: 31, baris: 20
  Merge pressing     → kolom: 38, baris: 20
  Merge sequences    → kolom: 46, baris: 20
  Merge misc         → kolom: 58, baris: 20

✅ Final merged:
   Jumlah klub  : 20
   Jumlah kolom : 58
   Kolom        : ['club', 'conversion_pct', 'goals', 'goals_vs_xg', 'played', 'shots', 'sot', 'xg', 'xg_per_shot', 'aerial_duels_won_pct', 'avg_possession_pct', 'blocks', 'clearances', 'ground_duels_won_pct', 'interceptions', 'possession_won', 'tackles', 'crosses_pct', 'crosses_successful', 'crosses_total', 'direction_bwd_pct', 'direction_fwd_pct', 'direction_left_pct', 'direction_right_pct', 'final_third_pct', 'final_third_successful', 'final_third_total', 'passes_pct', 'passes_successful', 'passes_total', 'through_balls', 'high_turnovers_goal_ending', 'high_turnovers_shot_ending', 'high_turnovers_shot_pct', 'high_turnovers_total', 'ppda', 'pressed_seqs', 'start_distance', 'buildups_goals', 'buildups_total', '

In [20]:
# ────────────────────────────────────────────────
# CELL 10 — Statistik Deskriptif
# ────────────────────────────────────────────────
print("📊 Statistik Deskriptif:\n")
df_merged.describe().show(truncate=False)

📊 Statistik Deskriptif:



26/07/09 17:47:44 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


+-------+----------+--------------------+------------------+--------------------+------+-----------------+-----------------+------------------+--------------------+--------------------+-------------------+------------------+------------------+--------------------+-----------------+-----------------+----------------+--------------------+------------------+------------------+-------------------+--------------------+-------------------+--------------------+-------------------+----------------------+------------------+--------------------+------------------+------------------+------------------+--------------------------+--------------------------+-----------------------+--------------------+------------------+-----------------+------------------+------------------+-----------------+--------------------+--------------------+-------------------+-----------------+------------------+------------------+-------------------+-------------------+-----------------+-----------------+----------------

In [21]:
# ────────────────────────────────────────────────
# CELL 11 — Deteksi Outlier (IQR Method)
# ────────────────────────────────────────────────
from pyspark.sql.types import NumericType
print("🔍 Kolom dengan outlier (IQR method):\n")
numeric_cols = [
    field.name for field in df_merged.schema.fields
    if isinstance(field.dataType, (FloatType, IntegerType))
       or field.dataType.simpleString() in ("double", "long", "bigint")
]
for col_name in numeric_cols:
    quantiles = df_merged.approxQuantile(col_name, [0.25, 0.75], 0.01)
    if len(quantiles) == 2:
        Q1, Q3 = quantiles
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        
        outliers = (
            df_merged
            .filter((F.col(col_name) < lower) | (F.col(col_name) > upper))
            .select("club")
            .rdd.flatMap(lambda x: x).collect()
        )
        if outliers:
            print(f"  ⚠️  {col_name:35s} → {outliers}")

🔍 Kolom dengan outlier (IQR method):

  ⚠️  goals                               → ['Barcelona']
  ⚠️  shots                               → ['Barcelona', 'Real Madrid']
  ⚠️  sot                                 → ['Barcelona', 'Real Madrid']
  ⚠️  xg                                  → ['Barcelona', 'Real Madrid']
  ⚠️  avg_possession_pct                  → ['Barcelona', 'Real Madrid']
  ⚠️  clearances                          → ['Real Madrid']
  ⚠️  ground_duels_won_pct                → ['Real Madrid', 'Athletic Bilbao']
  ⚠️  possession_won                      → ['Rayo Vallecano', 'Athletic Bilbao']
  ⚠️  direction_bwd_pct                   → ['Elche']
  ⚠️  direction_fwd_pct                   → ['Getafe']
  ⚠️  direction_left_pct                  → ['Getafe']
  ⚠️  direction_right_pct                 → ['Barcelona', 'Getafe']
  ⚠️  final_third_successful              → ['Barcelona', 'Real Madrid']
  ⚠️  final_third_total                   → ['Barcelona', 'Real Madrid']
  ⚠️  passes_

In [22]:
# ────────────────────────────────────────────────
# CELL 12 — Preview Final Dataset (Siap Silver)
# ────────────────────────────────────────────────
print("✅ Dataset siap untuk Silver Layer")
print(f"   Shape : ({df_merged.count()}, {len(df_merged.columns)})")
df_merged.show(20, truncate=False)

✅ Dataset siap untuk Silver Layer
   Shape : (20, 58)
+---------------+-------------------+-----+-----------+------+-----+---+-----+-----------+--------------------+-------------------+------+----------+--------------------+-------------+--------------+-------+-------------------+------------------+-------------+-------------------+-------------------+-------------------+-------------------+------------------+----------------------+-----------------+------------------+-----------------+------------+-------------+--------------------------+--------------------------+-----------------------+--------------------+----+------------+--------------+--------------+--------------+--------------------+--------------------+------------+--------------+--------------+-------------+-------------------+-------------------+------+-----+--------+-----------+-------------+--------+----+----------+---------+-------+
|club           |conversion_pct     |goals|goals_vs_xg|played|shots|sot|xg   |xg_per_shot

In [23]:
# Schema lengkap
df_merged.printSchema()

root
 |-- club: string (nullable = true)
 |-- conversion_pct: double (nullable = true)
 |-- goals: long (nullable = true)
 |-- goals_vs_xg: double (nullable = true)
 |-- played: long (nullable = true)
 |-- shots: long (nullable = true)
 |-- sot: long (nullable = true)
 |-- xg: double (nullable = true)
 |-- xg_per_shot: double (nullable = true)
 |-- aerial_duels_won_pct: double (nullable = true)
 |-- avg_possession_pct: double (nullable = true)
 |-- blocks: long (nullable = true)
 |-- clearances: long (nullable = true)
 |-- ground_duels_won_pct: double (nullable = true)
 |-- interceptions: long (nullable = true)
 |-- possession_won: long (nullable = true)
 |-- tackles: long (nullable = true)
 |-- crosses_pct: double (nullable = true)
 |-- crosses_successful: long (nullable = true)
 |-- crosses_total: long (nullable = true)
 |-- direction_bwd_pct: double (nullable = true)
 |-- direction_fwd_pct: double (nullable = true)
 |-- direction_left_pct: double (nullable = true)
 |-- direction_rig

In [24]:
spark.stop()